# Toxic Comment Preprocessing Runner (Google Colab)

This notebook runs the existing toxic comment preprocessing pipeline in Google Colab.

- Jigsaw is the mandatory main dataset for the non-extension experiment.
- Civil Comments is optional, extension-oriented, and disabled by default in this notebook.
- This notebook does not train models.
- The notebook focuses on preprocessing, validation, and output generation.

The existing Python package under `src/preprocessing/` remains the source of truth. This notebook imports and calls that implementation rather than changing or replacing it.

## 1. Colab Environment Setup

Install the preprocessing dependencies used by the repository. These packages match the current `requirements.txt` contents: pandas, numpy, scikit-learn, pyarrow, PyYAML, and transformers.

Colab runtimes often include some of these packages already, but this cell makes the notebook more reproducible.

In [ ]:
# Install project preprocessing dependencies in the Colab runtime.
# This uses explicit packages because requirements.txt is inside Drive and is mounted later.
%pip install -q pandas numpy scikit-learn pyarrow PyYAML transformers

# Confirm that the main packages import successfully.
import pandas as pd
import numpy as np
import sklearn
import pyarrow
import yaml
import transformers

print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('pyarrow:', pyarrow.__version__)
print('PyYAML:', yaml.__version__)
print('transformers:', transformers.__version__)

## 2. Mount Google Drive

The raw dataset folder is expected at:

`/content/drive/MyDrive/Colab Notebooks/toxic_comment_detection/dataset`

Civil Comments is controlled by the notebook-level `USE_CIVIL` switch. The default is `USE_CIVIL = False`, which runs Jigsaw-only preprocessing.

The notebook also assumes the repository source files are available under:

`/content/drive/MyDrive/Colab Notebooks/toxic_comment_detection`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define the Drive-backed project and dataset paths used by this notebook.
# Civil Comments is disabled by default so the notebook runs in Jigsaw-only mode.
# DATASET_ROOT is required by the assignment specification.
from pathlib import Path

USE_CIVIL = False

PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/toxic_comment_detection')
DATASET_ROOT = "/content/drive/MyDrive/Colab Notebooks/toxic_comment_detection/dataset"
DATASET_ROOT_PATH = Path(DATASET_ROOT)

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
METADATA_DIR = PROJECT_ROOT / 'metadata'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'preprocessing'

print('USE_CIVIL:', USE_CIVIL)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT_PATH)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('METADATA_DIR:', METADATA_DIR)
print('REPORT_DIR:', REPORT_DIR)

## 3. Repository Path Setup

Make the existing project package importable in the Colab runtime. This does not modify the source code. It only changes the notebook runtime working directory and Python import path.

In [ ]:
# Make the repository importable in Colab.
import os
import sys

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'PROJECT_ROOT does not exist: {PROJECT_ROOT}. '
        'Upload or sync the toxic_comment_detection repository to the expected Drive location.'
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Current working directory:', os.getcwd())
print('Project root is on sys.path:', str(PROJECT_ROOT) in sys.path)

In [ ]:
# Verify that the existing preprocessing package can be imported.
# The notebook uses this package as the source of truth.
from src.preprocessing.config import PipelineConfig
from src.preprocessing.pipeline import run_pipeline, setup_logging

print('Imported PipelineConfig and run_pipeline from src.preprocessing.')

## 4. Verify File Structure and Raw Inputs

The current pipeline expects the Jigsaw training file to be mandatory. Civil Comments is optional and controlled by `USE_CIVIL`.

Expected raw paths under the Drive dataset root:

- Jigsaw: `dataset/jigsaw/raw/train.csv`
- Civil Comments: `dataset/civil_comments/raw/train.csv`

The notebook fails clearly if Jigsaw is missing. When `USE_CIVIL = False`, Civil input is disabled and no Civil file is required. When `USE_CIVIL = True`, the Civil path is checked and passed to the existing pipeline.

In [ ]:
# Define raw input paths from the required Drive dataset root.
# CIVIL_RAW is None when USE_CIVIL is False.
JIGSAW_RAW = DATASET_ROOT_PATH / 'jigsaw' / 'raw' / 'train.csv'
CIVIL_RAW = DATASET_ROOT_PATH / 'civil_comments' / 'raw' / 'train.csv' if USE_CIVIL else None

# The existing preprocessing package expects a Path for civil_raw_path.
# When Civil is disabled, pass a deliberately missing path so the package uses
# its existing optional-missing-Civil behavior without source-code changes.
CIVIL_RAW_FOR_PIPELINE = (
    CIVIL_RAW
    if CIVIL_RAW is not None
    else PROJECT_ROOT / '.disabled_optional_inputs' / 'civil_comments' / 'raw' / 'train.csv'
)

print('USE_CIVIL:', USE_CIVIL)
print('Jigsaw raw:', JIGSAW_RAW)
print('Civil raw:', CIVIL_RAW)
print('Jigsaw exists:', JIGSAW_RAW.exists())
print('Civil enabled:', USE_CIVIL)
print('Civil exists:', CIVIL_RAW.exists() if CIVIL_RAW is not None else 'disabled')

if not JIGSAW_RAW.exists():
    raise FileNotFoundError(
        f'Mandatory Jigsaw raw file is missing: {JIGSAW_RAW}. '
        'Place train.csv at dataset/jigsaw/raw/train.csv under DATASET_ROOT.'
    )

if not USE_CIVIL:
    print('Civil Comments disabled. The pipeline will run in Jigsaw-only mode.')
elif CIVIL_RAW is not None and not CIVIL_RAW.exists():
    print('WARNING: USE_CIVIL=True but the Civil raw file is missing. The existing pipeline should continue in Jigsaw-only mode.')
else:
    print('Civil Comments raw file found. Optional extension artifacts will be generated.')

## 5. Preprocessing Stages Implemented by the Project

The existing project pipeline implements the following preprocessing stages:

1. Raw loading from CSV files.
2. Schema validation for expected text and toxicity label columns.
3. Initial data audit summaries.
4. Binary label harmonization.
5. Light, non-destructive text normalization.
6. Model-specific text views: `raw_text`, `text_clean`, and `text_tfidf`.
7. Deterministic stratified split generation based on `binary_label`.
8. Challenge slice construction for identity, obfuscation, implicit proxy, and length buckets.
9. Validation and leakage checks.
10. Metadata and preprocessing report generation.

This notebook runs those existing stages through the current Python package. It does not introduce new preprocessing rules.

## 6. Execute the Preprocessing Pipeline

This cell calls `run_pipeline(config)` from the existing `src.preprocessing.pipeline` module.

The configuration below points raw inputs at the required Google Drive dataset root while keeping output folders aligned with the repository layout. Civil Comments remains disabled unless `USE_CIVIL` is explicitly set to `True` near the top of the notebook.

- `data/processed/`
- `metadata/`
- `reports/preprocessing/`

The default configuration requires the DistilBERT tokenizer for token-length diagnostics. In Colab, this usually downloads through `transformers` if it is not cached.

In [ ]:
# Run the existing preprocessing pipeline with Colab/Drive paths.
# No source files under src/preprocessing are modified by this notebook.

setup_logging()

# Keep this False for full parity with the repository default.
# Set to True only if a teammate intentionally wants to continue without tokenizer diagnostics.
ALLOW_MISSING_TOKENIZER = False

config = PipelineConfig(
    project_root=PROJECT_ROOT,
    jigsaw_raw_path=JIGSAW_RAW,
    civil_raw_path=CIVIL_RAW_FOR_PIPELINE,
    processed_dir=PROJECT_ROOT / 'data' / 'processed',
    metadata_dir=PROJECT_ROOT / 'metadata',
    report_dir=PROJECT_ROOT / 'reports' / 'preprocessing',
    require_transformers_tokenizer=not ALLOW_MISSING_TOKENIZER,
)

report_payload = run_pipeline(config)

print('Preprocessing complete.')
print('Warnings:', report_payload.get('warnings', []))

## 7. Show Generated Outputs

List the generated processed datasets, metadata files, and report files. Then preview the teammate-facing CSV exports and richer split datasets if they exist.

In [ ]:
# List generated files in the output directories.
def list_files(path: Path):
    print(f'\n{path}')
    if not path.exists():
        print('  MISSING')
        return
    for item in sorted(path.iterdir()):
        if item.is_file():
            print(' ', item.name)

list_files(PROCESSED_DIR)
list_files(METADATA_DIR)
list_files(REPORT_DIR)

In [ ]:
# Preview selected output datasets.
# Prefer Parquet because it preserves types well; fall back to CSV if needed.
from IPython.display import display

def preview_dataset(stem: str, n: int = 3):
    parquet_path = PROCESSED_DIR / f'{stem}.parquet'
    csv_path = PROCESSED_DIR / f'{stem}.csv'
    if parquet_path.exists():
        df = pd.read_parquet(parquet_path)
        source_path = parquet_path
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
        source_path = csv_path
    else:
        print(f'{stem}: not found')
        return

    print(f'\n{source_path}')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    display(df.head(n))

preview_stems = [
    'train',
    'val',
    'test',
    'jigsaw_train',
    'jigsaw_val',
    'jigsaw_test',
]

if USE_CIVIL:
    preview_stems.extend([
    'civil_aug_pool_full',
    'civil_external_test',
    'civil_aug_pool_capped_match_jigsaw_train',
    'civil_aug_pool_identity',
    ])
else:
    print('Skipping Civil output previews because USE_CIVIL=False.')

for stem in preview_stems:
    preview_dataset(stem)

## 8. Preview Key Metadata and Reports

The pipeline writes machine-readable metadata and human-readable reports. This section previews the most useful generated files for handoff and verification.

In [ ]:
# Preview selected metadata JSON files.
import json
from pprint import pprint

def load_json(path: Path):
    if not path.exists():
        print(f'MISSING: {path}')
        return None
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

split_summary = load_json(METADATA_DIR / 'split_summary.json')
label_mapping = load_json(METADATA_DIR / 'label_mapping.json')
data_audit = load_json(METADATA_DIR / 'data_audit_summary.json')

print('\nSplit summary:')
pprint(split_summary)

print('\nLabel mapping:')
pprint(label_mapping)

print('\nData audit top-level keys:')
print(list(data_audit.keys()) if data_audit else None)

In [ ]:
# Show report paths and a short preview of the Markdown report.
report_md = REPORT_DIR / 'preprocessing_report.md'
report_json = REPORT_DIR / 'preprocessing_report.json'
handoff_md = REPORT_DIR / 'README_preprocessing.md'

print('Markdown report:', report_md, 'exists=', report_md.exists())
print('JSON report:', report_json, 'exists=', report_json.exists())
print('Handoff README:', handoff_md, 'exists=', handoff_md.exists())

if report_md.exists():
    print('\nFirst 80 lines of preprocessing_report.md:')
    lines = report_md.read_text(encoding='utf-8').splitlines()
    for line in lines[:80]:
        print(line)

## 9. Practical Notes for Teammates

Main non-extension experiment:

- Use only Jigsaw processed files:
  - `data/processed/train.csv`
  - `data/processed/val.csv`
  - `data/processed/test.csv`
  - `data/processed/jigsaw_train.parquet` or `.csv`
  - `data/processed/jigsaw_val.parquet` or `.csv`
  - `data/processed/jigsaw_test.parquet` or `.csv`
- The simplified teammate-facing CSVs contain exactly `raw_text`, `clean_text`, and `label`.
- Use `text_tfidf` for TF-IDF + Logistic Regression in the richer Jigsaw files.
- Use `text_clean` for DistilBERT.
- Use `binary_label` as the target.

Optional extension work:

- Civil Comments outputs are optional extension artifacts.
- Use Civil files only for controlled augmentation, robustness analysis, or external evaluation.
- Challenge slices are useful for later error analysis after model predictions exist.

This notebook is only for preprocessing and data preparation. It does not train or evaluate models.